# Base Python File

In [1]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns 

In [2]:
# Create timestamp and author
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print ("Current Time: {}".format(time_now))
print ("Andre Padilla Navarro")

Current Time: 09_27_2026_0821AM
Andre Padilla Navarro


In [3]:
# Read the datasets and store it as data2025 and data2026
data2025=pd.read_csv('2025.csv',encoding='utf-8')
data2026=pd.read_csv('2026PartYear.csv',encoding='utf-8')

In [4]:
# 2025-2026 new
new=pd.concat([data2026,data2025])

In [5]:
#Calculate base table (Periods 1–12)
new['Total Bulk'] = new['4046'] + new['4225'] + new['4770']
new['Year'] = pd.to_datetime(new['Current Year Week Ending']).dt.year

bulk_vs_bags = (new[new['Period'].between(1, 12)].groupby('Year')[['Total Bulk','TotalBagged','Total Bulk and Bags']].sum())

#Append YoY Growth percentage row
yoy_row = ((bulk_vs_bags.loc[2026] - bulk_vs_bags.loc[2025]) / bulk_vs_bags.loc[2025]) * 100
bulk_vs_bags.loc['YoY Growth (%)'] = yoy_row

#Format DataFrame (Integers for years, percentages for bottom row)
formatted_table = bulk_vs_bags.apply(lambda col: col.map(lambda x: f"{x:.2f}%" if col.name == col.name and isinstance(x, float) and x < 100 else f"{x:,.0f}"))

display(formatted_table)

,Total Bulk,TotalBagged,Total Bulk and Bags
Year,,,
2025,"1,117,462,691","811,226,582","1,928,689,274"
2026,"1,249,949,459","1,008,333,893","2,258,283,353"
YoY Growth (%),11.86%,24.30%,17.09%


In [6]:
#Filter Periods 1–12 and calculate Revenue
period = new[new['Period'].between(1, 12)].copy()
period['Revenue'] = period['ASP Current Year'] * period['Total Bulk and Bags']

#Aggregate Revenue & Volume by Year and Type
totals = period.groupby(['Year', 'Type'])[['Revenue', 'Total Bulk and Bags']].sum()
asp = (totals['Revenue'] / totals['Total Bulk and Bags']).unstack()

#Append YoY Growth (%)
asp.loc['YoY Growth (%)'] = ((asp.loc[2026] - asp.loc[2025]) / asp.loc[2025]) * 100

#Format output (uses .apply to ensure compatibility across pandas versions)
output = asp.iloc[:2].apply(lambda col: col.map(lambda x: f"${x:.2f}"))
output.loc['YoY Growth (%)'] = asp.loc['YoY Growth (%)'].map(lambda x: f"{x:+.2f}%")

display(output)

Type,Conventional,Organic
Year,,
2025,$1.22,$1.72
2026,$0.92,$1.53
YoY Growth (%),-24.28%,-11.03%


In [7]:
#Filter for Periods 1–12 and group by Year for PLU size codes
plu = (new[new['Period'].between(1, 12)].groupby('Year')[['4046', '4225', '4770']].sum())

# 2. Append YoY Growth (%)
plu.loc['YoY Growth (%)'] = ((plu.loc[2026] - plu.loc[2025]) / plu.loc[2025]) * 100

# 3. Format output (Volume counts with commas, YoY growth with sign and percentage)
table = plu.iloc[:2].apply(lambda col: col.map(lambda x: f"{x:,.0f}"))
table.loc['YoY Growth (%)'] = plu.loc['YoY Growth (%)'].map(lambda x: f"{x:+.2f}%")

display(table)

,4046,4225,4770
Year,,,
2025,"894,513,441","203,561,910","19,387,340"
2026,"891,935,307","306,088,070","51,926,083"
YoY Growth (%),-0.29%,+50.37%,+167.83%
